# RECON — Evidencia numérica (BI vs SQL)
**Objetivo:** generar evidencia reproducible (SQL ejecutado desde Python) para los KPIs críticos del dashboard **RECON** y dejar salidas en **CSV + XLSX + log**.

> Este notebook **NO** cambia Power BI. Solo produce evidencia externa (Postgres) para cerrar coherencia numérica ante tribunal.

## Qué genera (outputs)
- `outputs/recon_kpis_sql.csv` → resultados SQL (universo DW_SEM)
- `outputs/recon_kpis_bi_template.csv` → plantilla para pegar valores de Power BI (si no existe aún)
- `outputs/recon_kpis_bi_vs_sql.csv` → comparación BI vs SQL (differences + estado)
- `outputs/recon_bi_vs_sql_csv.xlsx` → resumen en Excel (hojas: SQL, BI, COMP)
- `outputs/recon_kpis_sql.log` → log con timestamp, conexión y SQL ejecutado

## KPIs cubiertos (contrato RECON)
- **LIC_Conteo** (dw_sem.v_fact_licitaciones_sem_v2)
- **OC_Conteo_etiqueta** (dw_sem.v_fact_ordenes_compra_sem)
- **OC_Conteo_Total_tabla** (dw_sem.v_fact_ordenes_compra_sem_v2)
- **OC_Monto_Total_etiqueta** (por moneda, dw_sem.v_fact_ordenes_compra_sem)
- **OC_Monto_Total_tabla** (por moneda, dw_sem.v_fact_ordenes_compra_sem_v2)
- **OC_Monto_Total_fecha_aceptacion_valida** (por moneda, `fecha_aceptacion_sk IS NOT NULL`, dw_sem.v_fact_ordenes_compra_sem_v2)

## Requisitos
- Acceso a Postgres del DW (`chile-pg` o servidor local).
- Variables de entorno recomendadas: `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, `PGPASSWORD`.


In [ ]:
# 0) Setup
# En Colab: descomenta si no tienes dependencias
# !pip -q install sqlalchemy psycopg2-binary pandas python-dotenv openpyxl

import os
import sys
import re
from datetime import datetime
from decimal import Decimal, InvalidOperation, getcontext

import pandas as pd

try:
    from sqlalchemy import create_engine, text
except Exception as e:
    raise RuntimeError("Falta sqlalchemy. Instala con: pip install sqlalchemy psycopg2-binary") from e

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    # opcional: si no existe python-dotenv, no pasa nada
    pass

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

# Precisión para montos grandes
getcontext().prec = 50

RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = os.getenv("OUT_DIR", "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("RUN_TS:", RUN_TS)
print("OUT_DIR:", os.path.abspath(OUT_DIR))


## 1) Conexión a Postgres (DW)
Este bloque usa variables de entorno si existen. **No** deja contraseña hardcodeada por defecto.


In [ ]:
# 1) Parámetros de conexión (env vars)
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5433")
PGDATABASE = os.getenv("PGDATABASE", "chilecompra")
PGUSER = os.getenv("PGUSER", "chile_user")
PGPASSWORD = os.getenv("PGPASSWORD","CHANGE_ME")

print("PGHOST:", PGHOST)
print("PGPORT:", PGPORT)
print("PGDATABASE:", PGDATABASE)
print("PGUSER:", PGUSER)
print("PGPASSWORD:", "***" if PGPASSWORD else "(vacío) -> define PGPASSWORD")

if not PGPASSWORD:
    raise RuntimeError("PGPASSWORD vacío. Define la variable de entorno PGPASSWORD o asígnala aquí (solo local).")

conn_str = f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}"
engine = create_engine(conn_str, pool_pre_ping=True)

# Smoke test
with engine.connect() as conn:
    conn.execute(text("SELECT 1;"))
print("Conexión OK.")


## 2) Catálogo de KPIs y SQL espejo
Definimos un **catálogo** para estandarizar: nombre KPI, tipo (conteo/monto), granularidad y SQL.


In [ ]:
# 2) Catálogo KPI RECON (contrato)
# - 'grain' = 'global' o 'moneda'
# - 'kpi_type' = 'count' o 'amount'
#
# Nota auditoría:
# - En RECON, los montos y conteos OC se filtran por slicer moneda_norm.
# - Por eso se incluyen KPIs por moneda_norm.
# - Se fuerza CAST a TEXT en montos para evitar notación científica en pandas.

CURRENCIES = ["CLP", "USD", "EUR", "CLF", "UTM"]

KPI_CATALOG = [

    # ============================================================
    # LIC (global)
    # ============================================================
    {
        "kpi": "LIC_Conteo",
        "kpi_type": "count",
        "grain": "global",
        "source_view": "dw_sem.v_fact_licitaciones_sem_v2",
        "sql": """
            SELECT COUNT(*)::bigint AS valor
            FROM dw_sem.v_fact_licitaciones_sem_v2;
        """
    },

    # ============================================================
    # OC (conteos)
    # ============================================================

    # Conteo global total (tabla completa)
    {
        "kpi": "OC_Conteo_Total_tabla",
        "kpi_type": "count",
        "grain": "global",
        "source_view": "dw_sem.v_fact_ordenes_compra_sem_v2",
        "sql": """
            SELECT COUNT(*)::bigint AS valor
            FROM dw_sem.v_fact_ordenes_compra_sem_v2;
        """
    },

    # Conteo por moneda_norm (para comparación con slicer)
    {
        "kpi": "OC_Conteo_por_moneda",
        "kpi_type": "count",
        "grain": "moneda",
        "source_view": "dw_sem.v_fact_ordenes_compra_sem_v2",
        "sql": """
            SELECT
                moneda_norm,
                COUNT(*)::bigint AS valor
            FROM dw_sem.v_fact_ordenes_compra_sem_v2
            GROUP BY moneda_norm
            ORDER BY moneda_norm;
        """
    },

    # Conteo por moneda_norm (etiqueta/tarjeta)
    # (en práctica es el mismo universo, pero se mantiene como KPI separado para auditoría formal)
    {
        "kpi": "OC_Conteo_etiqueta",
        "kpi_type": "count",
        "grain": "moneda",
        "source_view": "dw_sem.v_fact_ordenes_compra_sem_v2",
        "sql": """
            SELECT
                moneda_norm,
                COUNT(*)::bigint AS valor
            FROM dw_sem.v_fact_ordenes_compra_sem_v2
            GROUP BY moneda_norm
            ORDER BY moneda_norm;
        """
    },

    # ============================================================
    # OC (montos)
    # ============================================================

    # Monto total por moneda_norm (tabla)
    {
        "kpi": "OC_Monto_Total_tabla",
        "kpi_type": "amount",
        "grain": "moneda",
        "source_view": "dw_sem.v_fact_ordenes_compra_sem_v2",
        "sql": """
            SELECT
                moneda_norm,
                SUM(monto_total)::numeric(38,0)::text AS valor
            FROM dw_sem.v_fact_ordenes_compra_sem_v2
            GROUP BY moneda_norm
            ORDER BY moneda_norm;
        """
    },

    # Monto total por moneda_norm (etiqueta/tarjeta)
    # (mismo SQL, se mantiene separado para evidencia formal)
    {
        "kpi": "OC_Monto_Total_etiqueta",
        "kpi_type": "amount",
        "grain": "moneda",
        "source_view": "dw_sem.v_fact_ordenes_compra_sem_v2",
        "sql": """
            SELECT
                moneda_norm,
                SUM(monto_total)::numeric(38,0)::text AS valor
            FROM dw_sem.v_fact_ordenes_compra_sem_v2
            GROUP BY moneda_norm
            ORDER BY moneda_norm;
        """
    },

    # Monto total por moneda_norm con fecha_aceptacion válida
    {
        "kpi": "OC_Monto_Total_fecha_aceptacion_valida",
        "kpi_type": "amount",
        "grain": "moneda",
        "source_view": "dw_sem.v_fact_ordenes_compra_sem_v2",
        "sql": """
            SELECT
                moneda_norm,
                SUM(monto_total)::numeric(38,0)::text AS valor
            FROM dw_sem.v_fact_ordenes_compra_sem_v2
            WHERE fecha_aceptacion_sk IS NOT NULL
            GROUP BY moneda_norm
            ORDER BY moneda_norm;
        """
    },
]

# Vista catálogo resumida (para documentación)
df_catalog = pd.DataFrame(KPI_CATALOG)[["kpi", "kpi_type", "grain", "source_view"]]
df_catalog


## 3) Ejecutar SQL y capturar evidencia
Genera `recon_kpis_sql.csv` y `recon_kpis_sql.log` en `outputs/`.


In [ ]:
# 3) Helpers
def _to_decimal(x):
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and x.strip() == ""):
        return None
    try:
        return Decimal(str(x))
    except (InvalidOperation, ValueError, TypeError):
        return None

def fetch_df(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

rows = []
log_lines = [
    f"RECON KPIs SQL EVIDENCE - {RUN_TS}",
    "-"*80,
    f"PGHOST={PGHOST} PGPORT={PGPORT} PGDATABASE={PGDATABASE} PGUSER={PGUSER}",
    f"OUT_DIR={os.path.abspath(OUT_DIR)}",
    "-"*80
]

for item in KPI_CATALOG:
    kpi = item["kpi"]
    sql = item["sql"].strip()
    log_lines.append(f"\n## KPI: {kpi}")
    log_lines.append(f"source_view: {item['source_view']}")
    log_lines.append(sql)

    df = fetch_df(sql)

    if kpi == "OC_Monto_Total_fecha_aceptacion_valida":
        print(df)
        print(df.dtypes)

    # Normalización a formato largo
    if "moneda_norm" in df.columns:
        for _, r in df.iterrows():
            rows.append({
                "kpi": kpi,
                "kpi_type": item["kpi_type"],
                "grain": item["grain"],
                "moneda_norm": r["moneda_norm"],
                "valor_sql": _to_decimal(r["valor"]),
                "source_view": item["source_view"],
                "run_ts": RUN_TS
            })
    else:
        rows.append({
            "kpi": kpi,
            "kpi_type": item["kpi_type"],
            "grain": item["grain"],
            "moneda_norm": None,
            "valor_sql": _to_decimal(df.loc[0, "valor"]) if len(df) else None,
            "source_view": item["source_view"],
            "run_ts": RUN_TS
        })

    log_lines.append(df.to_string(index=False))

df_sql = pd.DataFrame(rows)

# casts amigables para lectura (sin perder precisión interna)
df_sql["valor_sql_str"] = df_sql["valor_sql"].astype(str)

df_sql


In [ ]:
# 3.1) Guardar evidencia SQL (CSV + LOG)
out_sql_csv = os.path.join(OUT_DIR, "recon_kpis_sql.csv")
out_log = os.path.join(OUT_DIR, "recon_kpis_sql.log")

df_sql.drop(columns=["valor_sql"], errors="ignore").to_csv(out_sql_csv, index=False)

with open(out_log, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

print("Generados:")
print(" -", out_sql_csv)
print(" -", out_log)


## 4) Valores Power BI
Para mantener el patrón de auditoría, los valores BI se ingresan vía CSV.

- Si **ya tienes** `recon_kpis_bi.csv`, el notebook lo carga.
- Si **no existe**, crea `recon_kpis_bi_template.csv` para que pegues los valores del dashboard RECON.

Formato esperado:
- `kpi` (exacto al catálogo)
- `moneda_norm` (vacío/NULL para conteos globales; CLP/USD/UTM/… para montos)
- `valor_pbi` (número en unidades completas, sin abreviaturas)


In [ ]:
# 4) Cargar o crear plantilla BI (robusta, por moneda)
BI_CSV = os.path.join(OUT_DIR, "recon_kpis_bi.csv")
BI_TEMPLATE = os.path.join(OUT_DIR, "recon_kpis_bi_template.csv")

def init_bi_template(path_out: str) -> pd.DataFrame:
    """Crea plantilla lista para pegar valores desde Power BI.
    - grain=global -> 1 fila (moneda_norm vacío)
    - grain=moneda -> 1 fila por moneda en CURRENCIES
    """
    rows = []
    for _, r in df_catalog.iterrows():
        if r["grain"] == "moneda":
            for m in CURRENCIES:
                rows.append({
                    "kpi": r["kpi"],
                    "kpi_type": r["kpi_type"],
                    "grain": r["grain"],
                    "source_view": r["source_view"],
                    "moneda_norm": m,
                    "valor_pbi": None,
                    "notas": "Pegar valor desde RECON (Power BI). Usar unidades completas (sin 'mil M')."
                })
        else:
            rows.append({
                "kpi": r["kpi"],
                "kpi_type": r["kpi_type"],
                "grain": r["grain"],
                "source_view": r["source_view"],
                "moneda_norm": None,
                "valor_pbi": None,
                "notas": "Pegar valor desde RECON (Power BI). Usar unidades completas."
            })
    tpl = pd.DataFrame(rows)
    tpl.to_csv(path_out, index=False)
    return tpl

def load_bi_csv(path_in: str) -> pd.DataFrame:
    df = pd.read_csv(path_in, dtype=str)
    # normaliza columnas mínimas
    expected = ["kpi","kpi_type","grain","source_view","moneda_norm","valor_pbi","notas"]
    for col in expected:
        if col not in df.columns:
            df[col] = None
    df = df[expected]
    return df

if os.path.exists(BI_CSV):
    df_bi = load_bi_csv(BI_CSV)
    print("Cargado:", BI_CSV)
elif os.path.exists(BI_TEMPLATE):
    df_bi = load_bi_csv(BI_TEMPLATE)
    print("Cargado (template ya existe):", BI_TEMPLATE)
else:
    df_bi = init_bi_template(BI_TEMPLATE)
    print("Creada plantilla:", BI_TEMPLATE)

df_bi.head(10)


In [ ]:
# 4.5) Normalización mínima + MERGE (crea df_sql_n, df_bi_n y df_cmp)

# --- df_sql_n (desde df_sql) ---
df_sql_n = df_sql.copy()

# claves limpias
df_sql_n["kpi"] = df_sql_n["kpi"].astype(str).str.strip()
df_sql_n["moneda_norm"] = df_sql_n["moneda_norm"].fillna("").astype(str).str.strip()

# asegurar strings para comparación
if "valor_sql_str" not in df_sql_n.columns and "valor_sql" in df_sql_n.columns:
    df_sql_n["valor_sql_str"] = df_sql_n["valor_sql"].astype(str)

# --- df_bi_n (desde df_bi) ---
df_bi_n = df_bi.copy()

# normaliza nombres de columnas por si vienen con espacios
df_bi_n.columns = [c.strip() for c in df_bi_n.columns]

# columnas mínimas esperadas (las que tu propio template define)
expected_cols = ["kpi","kpi_type","grain","source_view","moneda_norm","valor_pbi","notas"]
for c in expected_cols:
    if c not in df_bi_n.columns:
        df_bi_n[c] = None

df_bi_n = df_bi_n[expected_cols]

# claves limpias
df_bi_n["kpi"] = df_bi_n["kpi"].astype(str).str.strip()
df_bi_n["moneda_norm"] = df_bi_n["moneda_norm"].fillna("").astype(str).str.strip()

# --- CATALOGO normalizado (df_catalog) ---
df_cat_n = df_catalog.copy()
df_cat_n["kpi"] = df_cat_n["kpi"].astype(str).str.strip()

# IMPORTANTE:
# df_catalog NO tiene moneda_norm, así que hay que expandirlo según grain
rows = []
for _, r in df_cat_n.iterrows():
    if r["grain"] == "moneda":
        for m in CURRENCIES:
            rows.append({**r.to_dict(), "moneda_norm": m})
    else:
        rows.append({**r.to_dict(), "moneda_norm": ""})

df_cat_n = pd.DataFrame(rows)

# --- MERGE final: catálogo + SQL + BI ---
df_cmp = pd.merge(
    df_cat_n,
    df_sql_n[["kpi","moneda_norm","valor_sql_str","run_ts"]].copy(),
    on=["kpi","moneda_norm"],
    how="left"
)

df_cmp = pd.merge(
    df_cmp,
    df_bi_n[["kpi","moneda_norm","valor_pbi"]].copy(),
    on=["kpi","moneda_norm"],
    how="left"
)

# Derivados estándar que tu bloque #5 espera
df_cmp["valor_pbi_str"] = df_cmp["valor_pbi"].fillna("").astype(str)
df_cmp["valor_sql_str"] = df_cmp["valor_sql_str"].fillna("").astype(str)

print("df_cmp creado:", df_cmp.shape)
df_cmp.head(10)


## 5) Comparación BI vs SQL
Reglas:
- Conteos: **igualdad exacta**.
- Montos: **igualdad exacta por moneda** (sin conversión).
- Si BI falta (`valor_pbi` vacío) → `FALTA_PBI`.
- Si SQL falta (no debería) → `FALTA_SQL`.


In [ ]:
# 5) Normalización BI + comparación (robusta: separadores miles/decimal, Decimal, NaN)
import re
import math
import numpy as np
import pandas as pd
from decimal import Decimal, InvalidOperation

# --- Tolerancias (auditoría) ---
TOL_COUNT_ABS  = Decimal("0")   # conteos deben calzar exacto
TOL_AMOUNT_ABS = Decimal("1")   # montos: tolerancia 1 unidad (por redondeos/casting)

# --- Guard rail: df_cmp debe existir (creado en la celda MERGE) ---
if "df_cmp" not in globals():
    raise RuntimeError("df_cmp no existe. Ejecuta primero la celda de MERGE (catálogo+SQL+BI).")

def normalize_number_str(x) -> str:
    """
    Normaliza formatos típicos:
    - '4.633.081' -> '4633081'                  (miles con puntos)
    - '27.651.861.086.540,85' -> '27651861086540.85'  (coma decimal)
    - '27651861086540.85' -> '27651861086540.85'      (punto decimal)
    Rechaza abreviados tipo 'mil', 'mill', 'mil M', etc.
    """
    if x is None:
        return ""
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return ""

    s_low = s.lower()
    if re.search(r"\bmil\b", s_low) or re.search(r"\bmill", s_low) or re.search(r"mil\s*m", s_low):
        return ""  # no auditable

    s = s.replace(" ", "")

    # Caso 1: coma decimal (estilo es-CL/es-ES) -> quitar puntos de miles y cambiar coma a punto
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
        return s

    # Caso 2: NO hay coma. Puede ser:
    # - miles con puntos: 4.633.081 (muchos puntos)
    # - decimal con punto: 27651861086540.85 (un punto, 1-2 decimales)
    if s.count(".") == 1:
        left, right = s.split(".")
        if right.isdigit() and len(right) in (1, 2) and left.isdigit():
            return s  # punto decimal real
        # si no cumple, tratarlo como miles
        return s.replace(".", "")

    # Caso 3: múltiples puntos -> miles
    return s.replace(".", "")

def to_decimal_safe(x):
    """Convierte a Decimal de forma segura. None si vacío/NaN/no numérico."""
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    # strings
    if isinstance(x, str):
        s = normalize_number_str(x)
        if s == "":
            return None
        try:
            d = Decimal(s)
            if d.is_nan() or d.is_infinite():
                return None
            return d
        except Exception:
            return None

    # numéricos
    try:
        d = Decimal(str(x))
        if d.is_nan() or d.is_infinite():
            return None
        return d
    except Exception:
        return None


def _ensure_decimal(x):
    """Garantiza Decimal limpio o None (sin NaN/inf)."""
    if isinstance(x, Decimal):
        if x.is_nan() or x.is_infinite():
            return None
        return x
    return to_decimal_safe(x)


def compute_row(row):
    """
    Devuelve (diff_abs, diff_pct, estado)
    estado: OK / NO_OK / FALTA_SQL / FALTA_PBI
    """
    a = _ensure_decimal(row.get("valor_pbi_dec"))
    b = _ensure_decimal(row.get("valor_sql_dec"))

    if b is None:
        return (None, None, "FALTA_SQL")
    if a is None:
        return (None, None, "FALTA_PBI")

    try:
        diff_abs = a - b
        diff_pct = (diff_abs / b) if b != 0 else None

        ktype = str(row.get("kpi_type") or "")
        tol = TOL_AMOUNT_ABS if ktype == "amount" else TOL_COUNT_ABS

        # defensa por valores raros
        if isinstance(diff_abs, Decimal) and (diff_abs.is_nan() or diff_abs.is_infinite()):
            return (None, None, "NO_OK")

        estado = "OK" if abs(diff_abs) <= tol else "NO_OK"
        return (diff_abs, diff_pct, estado)

    except (InvalidOperation, ZeroDivisionError):
        return (None, None, "NO_OK")


# --- Normaliza BI (si ya existe valor_pbi_str; si no, intenta derivarlo) ---
if "valor_pbi_str" not in df_cmp.columns and "valor_pbi" in df_cmp.columns:
    df_cmp["valor_pbi_str"] = df_cmp["valor_pbi"].astype(str).where(df_cmp["valor_pbi"].notna(), "")

if "valor_pbi_str" in df_cmp.columns:
    df_cmp["valor_pbi_str"] = df_cmp["valor_pbi_str"].replace({"nan": ""})
    df_cmp["valor_pbi_dec"] = df_cmp["valor_pbi_str"].apply(to_decimal_safe)
else:
    # si no existe, igual crea la columna para que compute_row marque FALTA_PBI
    df_cmp["valor_pbi_dec"] = None
    df_cmp["valor_pbi_str"] = ""


# --- Normaliza SQL (si ya existe valor_sql_str; si no, intenta derivarlo) ---
if "valor_sql_str" not in df_cmp.columns and "valor_sql" in df_cmp.columns:
    df_cmp["valor_sql_str"] = df_cmp["valor_sql"].astype(str).where(df_cmp["valor_sql"].notna(), "")

if "valor_sql_str" in df_cmp.columns:
    df_cmp["valor_sql_str"] = df_cmp["valor_sql_str"].replace({"nan": ""})
    df_cmp["valor_sql_dec"] = df_cmp["valor_sql_str"].apply(to_decimal_safe)
else:
    df_cmp["valor_sql_dec"] = None
    df_cmp["valor_sql_str"] = ""


# --- Comparación final ---
vals = df_cmp.apply(lambda r: compute_row(r), axis=1, result_type="expand")
df_cmp["diff_abs"] = vals[0]
df_cmp["diff_pct"] = vals[1]
df_cmp["estado"] = vals[2]

# --- Strings finales (para export) ---
df_cmp["diff_abs_str"] = df_cmp["diff_abs"].apply(lambda x: "" if x is None else str(x))
df_cmp["diff_pct_str"] = df_cmp["diff_pct"].apply(lambda x: "" if x is None else str(x))

df_cmp


## 6) Exportar evidencia (CSV + XLSX)
Genera:
- `recon_kpis_bi_vs_sql.csv`
- `recon_bi_vs_sql_csv.xlsx` (SQL / BI / COMP)


In [ ]:
# 6) Exportar evidencia (CSV + XLSX) — sin KeyError (reindex seguro)
out_cmp_csv = os.path.join(OUT_DIR, "recon_kpis_bi_vs_sql.csv")
out_xlsx    = os.path.join(OUT_DIR, "recon_bi_vs_sql_csv.xlsx")
out_sql_csv = os.path.join(OUT_DIR, "recon_kpis_sql.csv")
out_log     = os.path.join(OUT_DIR, "recon_kpis_sql.log")

# Export SQL normalizado (para trazabilidad)
df_sql_n.to_csv(out_sql_csv, index=False)

# Export comparación (columnas en orden fijo, aunque falte alguna -> se crea vacía)
export_cols = [
    "kpi","moneda_norm","kpi_type","grain","source_view","run_ts",
    "valor_pbi_str","valor_sql_str","diff_abs_str","diff_pct_str","estado"
]
df_cmp_export = df_cmp.reindex(columns=export_cols)
df_cmp_export.to_csv(out_cmp_csv, index=False)

# Log (SQL ejecutado + entorno)
with open(out_log, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

# Excel (3 hojas: SQL / BI / COMP)
try:
    import openpyxl  # noqa: F401
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
        df_sql_n.to_excel(w, index=False, sheet_name="SQL")
        df_bi_n.to_excel(w, index=False, sheet_name="BI")
        df_cmp_export.to_excel(w, index=False, sheet_name="COMP")
    excel_msg = f" - {out_xlsx}"
except Exception as e:
    excel_msg = f" (XLSX no generado: {e})"

print("Generados:")
print(" -", out_sql_csv)
print(" -", out_cmp_csv)
print(" -", out_log)
print(excel_msg)


## 7) Dictamen automático (para pegar en auditoría)
El dictamen se considera **CERRADO** solo si todo está `OK` (sin `NO_OK` y sin `FALTA_*`).


In [ ]:
# 7) Dictamen automático (para pegar en auditoría)
ok = int((df_cmp["estado"]=="OK").sum())
no_ok = int((df_cmp["estado"]=="NO_OK").sum())
falta_pbi = int((df_cmp["estado"]=="FALTA_PBI").sum())
falta_sql = int((df_cmp["estado"]=="FALTA_SQL").sum())

print("Resumen dictamen RECON")
print("- OK:", ok)
print("- NO_OK:", no_ok)
print("- Falta valor Power BI:", falta_pbi)
print("- Falta evidencia SQL:", falta_sql)

if no_ok == 0 and falta_pbi == 0 and falta_sql == 0:
    print("\n✅ Dictamen: coherencia numérica RECON CERRADA (BI vs SQL).")
else:
    print("\n⚠️ Dictamen: coherencia RECON NO CERRADA. Revisar filas NO_OK/FALTA_* (ver tabla siguiente).")
    display(df_cmp[df_cmp["estado"]!="OK"])
